In [0]:
# ============================================================
# MVP VTI CBL
# Notebook: 02_Silver
#
# Objetivo:
# Realizar limpeza, tipagem, padronização e normalização
# espacial das três fontes da camada Bronze.
#
# Fontes:
# - bronze_cbl_raw
# - bronze_locais_raw
# - bronze_blocos_raw
#
# Saídas:
# - silver_cbl
# - silver_locais
# - silver_blocos
# ============================================================

In [0]:
# Importa as funções do PySpark utilizadas no tratamento e padronização dos dados
from pyspark.sql import functions as F
# Importa Window para operações analíticas que possam ser utilizadas durante o tratamento
from pyspark.sql.window import Window

In [0]:
# Carrega as três tabelas Bronze que serão tratadas e padronizadas na camada Silver
df_cbl = spark.table("mvp_vti.bronze.bronze_cbl_raw")
df_locais = spark.table("mvp_vti.bronze.bronze_locais_raw")
df_blocos = spark.table("mvp_vti.bronze.bronze_blocos_raw")
# Confere se as três fontes foram carregadas corretamente antes de iniciar as transformações
print("CBL:", df_cbl.count())
print("Locais:", df_locais.count())
print("Blocos:", df_blocos.count())

CBL: 198239
Locais: 797
Blocos: 78160


In [0]:
# Filtra a base VTI mantendo somente as medições referentes ao CBL
# O tratamento com trim e upper evita diferenças causadas por espaços ou variações de escrita
df_cbl_filtrado = (
df_cbl
.filter(F.upper(F.trim(F.col("type"))) == "CBL-PEAK")
)



In [0]:
# Seleciona os campos da fonte VTI necessários para o tratamento do CBL e para as etapas seguintes do pipeline
df_cbl_filtrado = df_cbl_filtrado.select(
"id",
"subdivision",
"linecode",
"track_no",
"km_major",
"kp_minor",
"latitude",
"longitude",
"date_time",
"type",
"value",
"severity",
"vehicle",
"speed",
"direction",
"_ingestion_ts",
"_source"
)

In [0]:
# Padroniza o código da linha ferroviária para evitar diferenças de escrita nos relacionamentos posteriores
# Também ajusta os tipos dos campos numéricos e temporais utilizados nas etapas Silver e Gold
df_cbl_silver = (
df_cbl_filtrado
.withColumn("linecode", F.upper(F.trim(F.col("linecode"))))
.withColumn("km_major", F.col("km_major").cast("int"))
.withColumn("kp_minor", F.col("kp_minor").cast("int"))
.withColumn("value", F.col("value").cast("double"))
.withColumn("speed", F.col("speed").cast("double"))
.withColumn("date_time", F.to_timestamp("date_time"))
)

In [0]:
# Converte a referência KM + metro em uma posição ferroviária única, expressa em metros acumulados
# Exemplo: km_major = 7 e kp_minor = 145 resultam em posicao_m = 7145
df_cbl_silver = (
df_cbl_silver
.withColumn(
"posicao_m",
(F.col("km_major") * 1000 + F.col("kp_minor")).cast("long")
)
)


In [0]:
# Calcula uma referência matemática de bloco de 25 m a partir da posição acumulada do CBL
# Esta referência auxilia na validação espacial antes do relacionamento com a base oficial de blocos
df_cbl_silver = (
df_cbl_silver
.withColumn(
"bloco_25m_calc",
F.floor(F.col("posicao_m") / 25).cast("long")
)
.withColumn(
"bloco_inicio_calc_m",
F.col("bloco_25m_calc") * 25
)
.withColumn(
"bloco_fim_calc_m",
F.col("bloco_inicio_calc_m") + 25
)
)


In [0]:
# Padroniza o nome da chave ferroviária utilizada nas três fontes do projeto
# A coluna linecode da base VTI passa a utilizar a mesma nomenclatura das bases de Locais e Blocos
df_cbl_silver = df_cbl_silver.withColumnRenamed("linecode", "linha")

In [0]:
# Confere a quantidade total de registros e o volume de nulos nos campos essenciais da Silver CBL
# Esta verificação garante que os principais atributos utilizados nos relacionamentos espaciais estejam preenchidos
display(
df_cbl_silver.select(
F.count("*").alias("total_registros"),
F.sum(F.col("linha").isNull().cast("int")).alias("nulos_linha"),
F.sum(F.col("km_major").isNull().cast("int")).alias("nulos_km"),
F.sum(F.col("kp_minor").isNull().cast("int")).alias("nulos_metro"),
F.sum(F.col("date_time").isNull().cast("int")).alias("nulos_data_hora"),
F.sum(F.col("value").isNull().cast("int")).alias("nulos_cbl"),
F.sum(F.col("posicao_m").isNull().cast("int")).alias("nulos_posicao")
)
)


total_registros,nulos_linha,nulos_km,nulos_metro,nulos_data_hora,nulos_cbl,nulos_posicao
42881,0,1140,1140,0,0,1140


In [0]:
# Após o filtro CBL-PEAK, o campo genérico value passa a representar diretamente o valor de CBL
# A renomeação deixa a variável mais clara para os relacionamentos, agregações e análises posteriores
df_cbl_silver = df_cbl_silver.withColumnRenamed("value", "cbl")


In [0]:
# Padroniza a chave da linha ferroviária para manter consistência com as demais bases do projeto
# Converte os campos de referência em metros acumulados, criando o intervalo espacial do local
df_locais_silver = (
df_locais
.withColumn("linha", F.upper(F.trim(F.col("linha"))))
.withColumn("inicio_m", F.col("ponto_partida").cast("long"))
.withColumn("fim_m", F.col("ponto_final").cast("long"))
)

In [0]:
# Padroniza a identificação da linha ferroviária para manter consistência entre as fontes
# Converte os limites dos blocos de quilômetros decimais para metros acumulados
df_blocos_silver = (
df_blocos
.withColumn("linha", F.upper(F.trim(F.col("linha"))))
.withColumn("inicio_m", F.round(F.col("km_inicio") * 1000).cast("long"))
.withColumn("fim_m", F.round(F.col("km_fim") * 1000).cast("long"))
)


In [0]:
# Cria um identificador para cada bloco a partir da linha e dos limites do intervalo em metros
# O bloco_id facilita a identificação e o agrupamento dos segmentos nas etapas Gold e de análise
df_blocos_silver = (
df_blocos_silver
.withColumn(
"bloco_id",
F.concat_ws("_", F.col("linha"), F.col("inicio_m"), F.col("fim_m"))
)
)

In [0]:
# Calcula a extensão de cada bloco a partir dos limites espaciais já convertidos para metros
df_blocos_silver = (
df_blocos_silver
.withColumn("extensao_m", F.col("fim_m") - F.col("inicio_m"))
)
# Verifica a distribuição das extensões para confirmar a estrutura dos blocos utilizados no MVP
display(
df_blocos_silver
.groupBy("extensao_m")
.count()
.orderBy("extensao_m")
)

extensao_m,count
25,78160


In [0]:
# Persiste a base CBL tratada e normalizada como tabela Delta na camada Silver
# O modo overwrite permite reconstruir a tabela sempre que o pipeline for executado novamente
(
df_cbl_silver.write
.format("delta")
.mode("overwrite")
.saveAsTable("mvp_vti.silver.silver_cbl")
)

In [0]:
# Persiste a base de Locais já padronizada e preparada para os relacionamentos espaciais
# O modo overwrite permite reconstruir a tabela a cada nova execução do pipeline
(
df_locais_silver.write
.format("delta")
.mode("overwrite")
.saveAsTable("mvp_vti.silver.silver_locais")
)

In [0]:
# Persiste a base oficial de Blocos de 25 m já normalizada espacialmente na camada Silver
# Esta tabela será utilizada posteriormente para associar cada medição CBL ao seu respectivo bloco
(
df_blocos_silver.write
.format("delta")
.mode("overwrite")
.saveAsTable("mvp_vti.silver.silver_blocos")
)